# <font color="steelblue">14. Tasas de variación e índices simples</font>

*Unidad 3. Números índice y series cronológicas* · Sesión teórica 14 de 30

**Autoría**:

*   Fernando Borrás Rocher (f.borras@umh.es), autor de la idea original
*   Juan Francisco Monge Ivars
*   José Santiago García Cremades

Departamento de Estadística, Matemáticas e Informática. Universidad Miguel Hernández de Elche.

Basado en [SIMUSTAT](https://unidigitalsimustat.umh.es/) (proyecto UNIDIGITAL-UMH, CC BY-SA 4.0).

**Asignatura**: Estadística (1200), 1.º curso, Grado en Periodismo.

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="CC BY-SA 4.0" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a> Creative Commons Atribución-CompartirIgual 4.0.

> ⚠️ **Los datos de este curso son simulados.** Imitan la forma de fuentes reales (INE, CIS, audímetros, agencias de verificación), pero ninguna cifra es real. Úsalos para aprender; para publicar, ve siempre a la fuente oficial.

## <font color='steelblue'>Antes de empezar: configura el cuaderno</font>

Ejecuta **una vez** la celda de abajo (botón ▶). Instala la librería de las preguntas, descarga los datos (`paro_mensual.csv`) y las preguntas de autoevaluación. Si todo va bien verás **`Cuaderno configurado`**.

> 💡 Si Colab se desconecta, los ficheros se borran: vuelve a ejecutar esta celda.

In [ ]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración { display-mode: "form" }
import os, sys, json, base64, subprocess, urllib.request

RAW = "https://raw.githubusercontent.com/fborrasumh/statperiodismo/main/"
DATOS = ["paro_mensual.csv"]
PREGUNTAS = ["q_P14_1.json", "q_P14_2.json", "q_P14_3.json", "q_P14_4.json"]

try:
    from jupyterquiz import display_quiz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyterquiz==2.9.6.4"], check=True)
    from jupyterquiz import display_quiz

for carpeta, ficheros in (("datos/", DATOS), ("autoeval/", PREGUNTAS)):
    for f in ficheros:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + carpeta + f, f)

def pregunta(fichero):
    """Muestra una pregunta interactiva de autoevaluación."""
    display_quiz(json.loads(base64.b64decode(open(fichero, "rb").read()).decode("utf8")))

print("Cuaderno configurado")

## <font color='steelblue'>Introducción</font>

«El paro baja un 2 %», «la factura de la luz sube un 30 %», «el IPC se modera». Las tasas de variación y los números índice son el idioma de la información económica. Mal usados, producen los errores más repetidos de la prensa: confundir puntos con porcentajes, o celebrar una subida que solo es un **efecto base**.

**Qué necesitas saber antes**: los cuadernos 2 y 4.

## <font color='steelblue'>Qué vas a aprender</font>

Al terminar este cuaderno sabrás:

1. Calcular tasas de variación entre dos periodos y diferenciar puntos de porcentajes.
2. Construir un índice simple con base 100.
3. Calcular la tasa media anual acumulativa.
4. Reconocer el efecto base.

## <font color='steelblue'>Contenidos</font>

1. Tasas de variación
2. Puntos frente a porcentaje
3. Índices simples
4. Tasa media anual y efecto base

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 20)

## <font color='steelblue'>1. Tasas de variación</font>

In [ ]:
paro = pd.read_csv("paro_mensual.csv")
paro.tail()

La **tasa de variación** entre un periodo inicial (0) y uno final (t) es:

$$TV = \frac{x_t - x_0}{x_0} \times 100$$

Con Pandas, `pct_change()` la calcula respecto a la fila anterior, y `pct_change(12)` respecto a 12 filas antes (el mismo mes del año anterior: **tasa interanual**).

In [ ]:
paro["tasa_mensual"] = (paro["parados"].pct_change() * 100).round(2)
paro["tasa_interanual"] = (paro["parados"].pct_change(12) * 100).round(2)
paro.tail(6)

**Cómo leerlo**: la tasa mensual compara con el mes anterior (y se ve muy afectada por la temporada); la interanual compara con el mismo mes del año anterior y es la que se usa para hablar de **tendencia**.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escribe el número y pulsa Enter:

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P14_1.json")

## <font color='steelblue'>2. Puntos frente a porcentaje</font>

Cuando la variable **ya es un porcentaje** (tasa de paro, cuota de pantalla, intención de voto), hay dos formas de medir el cambio:

- **Diferencia en puntos porcentuales**: 12 % → 15 % son **3 puntos**.
- **Tasa de variación**: (15 − 12) / 12 = **25 %**.

Las dos son correctas, pero no intercambiables. «La tasa de paro sube un 3 %» cuando ha pasado del 12 % al 15 % es un error.

In [ ]:
antes, despues = 12.0, 15.0
print("Diferencia:", despues - antes, "puntos")
print("Tasa de variación:", round((despues - antes) / antes * 100, 1), "%")

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P14_2.json")

## <font color='steelblue'>3. Índices simples</font>

Un **índice simple** compara cada valor con el de un periodo de referencia (**periodo base**), al que se da el valor 100:

$$I_t = \frac{x_t}{x_{base}} \times 100$$

Un índice de 108 significa un 8 % más que en la base; uno de 95, un 5 % menos. Permite comparar series de tamaños muy distintos en la misma escala.

In [ ]:
paro["anio"] = paro["fecha"].str[:4].astype(int)
anual = paro.groupby("anio")["parados"].mean().round()
indice = (anual / anual.loc[2016] * 100).round(1)
pd.DataFrame({"parados (media anual)": anual, "índice 2016 = 100": indice})

In [ ]:
indice.plot(marker="o", color="steelblue")
plt.axhline(100, color="grey", lw=0.8)
plt.title("Paro registrado, índice 2016 = 100 (datos simulados)")
plt.ylabel("Índice")
plt.show()

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P14_3.json")

## <font color='steelblue'>4. Tasa media anual y efecto base</font>

Para resumir el crecimiento de varios años no se promedian las tasas: se usa la **tasa media anual acumulativa**:

$$TMA = \left(\frac{x_{final}}{x_{inicial}}\right)^{1/n} - 1$$

donde $n$ es el número de años transcurridos.

In [ ]:
ini, fin = anual.loc[2016], anual.loc[2025]
n = 2025 - 2016
tma = ((fin / ini) ** (1 / n) - 1) * 100
print(f"Tasa media anual 2016-2025: {tma:.2f} %")

El **efecto base** aparece cuando el periodo de comparación fue excepcional. Si el paro se disparó en 2020, la tasa interanual de 2021 parecerá una gran mejora aunque solo se vuelva a la normalidad. Antes de titular «récord», mira **contra qué** se compara.

In [ ]:
ventana = paro[(paro["fecha"] >= "2020-01") & (paro["fecha"] <= "2022-06")]
plt.plot(ventana["fecha"], ventana["tasa_interanual"], marker=".")
plt.axhline(0, color="grey", lw=0.8)
plt.xticks(ventana["fecha"][::3], rotation=45)
plt.title("Tasa interanual del paro: el salto de 2020 y su efecto base en 2021")
plt.ylabel("%")
plt.show()

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P14_4.json")

## <font color='steelblue'>Reto aprendiz</font>

Resuelve estos ejercicios en la celda vacía de debajo de cada enunciado. Si te atascas, vuelve a la sección indicada. La solución está plegada: **inténtalo antes de mirarla**.

### **Ejercicio 1** *(sección 1)*

¿Cuál fue el mes con la mayor caída mensual del paro en toda la serie?

<details><summary>Ver solución</summary>

```python
paro.loc[paro["tasa_mensual"].idxmin(), ["fecha", "tasa_mensual"]]
```
</details>

### **Ejercicio 2** *(sección 2)*

La cuota de pantalla de un informativo pasa del 14,5 % al 11,6 %. Escribe la frase correcta en puntos y en porcentaje.

<details><summary>Ver solución</summary>

```python
print(11.6 - 14.5, "puntos")
print(round((11.6 - 14.5) / 14.5 * 100, 1), "%")
```

Baja 2,9 puntos, es decir, un 20 %.
</details>

### **Ejercicio 3** *(sección 3)*

Construye el índice de paro con base 2019 = 100 y di cuánto ha cambiado el paro entre 2019 y 2025.

<details><summary>Ver solución</summary>

```python
(anual / anual.loc[2019] * 100).round(1)
```
</details>

### **Ejercicio 4** *(sección 4)*

Un diario digital pasó de 2,1 millones de usuarios únicos en 2020 a 3,4 millones en 2024. Calcula la tasa media anual acumulativa.

<details><summary>Ver solución</summary>

```python
((3.4 / 2.1) ** (1 / 4) - 1) * 100
```

Algo menos del 13 % anual. Promediar las tasas de cada año daría otro resultado.
</details>

## <font color='steelblue'>Resumen: ideas clave</font>

- **Tasa de variación** = (final − inicial) / inicial × 100; `pct_change()` la calcula en Pandas.
- La **tasa interanual** (`pct_change(12)` en datos mensuales) elimina el efecto de la temporada.
- Si la variable ya es un %, distingue **puntos** de **porcentaje de variación**.
- Un **índice simple** divide cada valor por el del periodo base y multiplica por 100.
- Resume varios años con la **tasa media anual acumulativa** y vigila el **efecto base**.

## <font color='steelblue'>Glosario</font>

| Término | Significado |
| :-- | :-- |
| **Tasa de variación** | Cambio relativo entre dos periodos. |
| **Tasa interanual** | Tasa respecto al mismo periodo del año anterior. |
| **Punto porcentual** | Diferencia entre dos porcentajes. |
| **Índice simple** | Valor relativo a un periodo base que vale 100. |
| **Efecto base** | Distorsión de una tasa por la excepcionalidad del periodo de comparación. |

## <font color='steelblue'>Errores frecuentes y cómo resolverlos</font>

| Síntoma | Causa probable | Solución |
| :-- | :-- | :-- |
| La primera fila de `pct_change()` es NaN | No hay periodo anterior | Es lo esperado; no es un error |
| `NameError: name 'pd' is not defined` | No se ejecutó la celda de importación | Ejecuta las celdas desde el principio, en orden |
| `FileNotFoundError` al leer un fichero | No se ejecutó la configuración o Colab se reinició | Ejecuta la celda *Configura el cuaderno* |
| La pregunta de autoevaluación no aparece | Falta la configuración | Ejecuta la celda de configuración y luego la pregunta |

## <font color='steelblue'>Para seguir</font>

En el cuaderno 15 construirás índices que combinan varios precios: Laspeyres y Paasche.

**Siguiente cuaderno**: [15. Índices compuestos: la cesta de la compra (Laspeyres y Paasche)](https://colab.research.google.com/github/fborrasumh/statperiodismo/blob/main/cuadernos/P15_indices_compuestos.ipynb)